In [59]:
"""
Day 1 - API Data Analysis Using Python Structures
Squad Python Specialization - Data Science, AI & Analytics
"""

# imports section
import json
from pathlib import Path
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import pandas as pd
import requests

In [ ]:
# url link to the data
API_URL = "https://jsonplaceholder.typicode.com/users"

In [8]:
# setting connection to the url and saving data to a variable

def get_customers(url: str = API_URL) -> list[dict]:
    """Fetch the customer list. Report failures clearly instead of crashing."""
    try:
        response = requests.get(url, timeout=10)
        # since data is json format
        data = response.json()
    # catching timeout error
    except requests.exceptions.Timeout:
        print(f"Error request timed out!")
        # if no connection, return empty list
        return []
    # catching connection error 
    except requests.exceptions.ConnectionError:
        print(f"Error could not connect!")
        return []
    
    return data

# saving data from url to customers variable
customers = get_customers()
# printing customers to view the data
print(customers)
    

[{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'address': {'street': 'Victor Plains', 'suite': 'Suite 879', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'geo': {'lat': '-43.9509', 'lng': '-34.4618'}}, 'phone': '010-692-6593 x09125', 'website': 'anastasia.net', 'company': {'name': 'Deckow-Crist', 'catchPhrase': 'Proactive didactic contingency', 'bs': 'synergize scalable supply-chains'}}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'address': {'street': 'Douglas Exten

In [ ]:
def explore(record: dict, indent: int = 0) -> None:
    """Print every key in a (possibly nested) record with its type, indented by depth."""
    
    # add gap to create indent showing hierachy 
    gap = "    " * indent

    for key, value in record.items():
        print(f"{gap}{key}: {type(value).__name__}")
        # catch where value is a dictionary, add indent 
        if isinstance(value, dict):
            explore(value, indent + 1)

#  testing explore with a customer
explore(customers[5])

id: int
name: str
username: str
email: str
address: dict
    street: str
    suite: str
    city: str
    zipcode: str
    geo: dict
        lat: str
        lng: str
phone: str
website: str
company: dict
    name: str
    catchPhrase: str
    bs: str


---

In [17]:
# function to extract email domain from the email

def _email_domain(email) -> str | None:
    # validating email by checking for the @ symbol
    if isinstance(email, str) and "@" in email:
        # extracting email domain
        # split email at last @, max 1 split, 
        # return second element at index 1 ie domain
        return email.rsplit("@", 1)[1]
    return None

In [ ]:
def extract_attributes(customers: list[dict]) -> list[dict]:
    """Return one flat dict per customer with the attributes listed in the task."""
    rows = []

    # looping through each customer in customers
    for c in customers:
        # setting email first since its used in etracting email domain
        email = c.get("email")

        rows.append({
            "id": c.get("id"),
            "name": c.get("name"),
            "username": c.get("username"),
            "email": email,
            # using _email_domain function to extract domain from email
            "email_domain": _email_domain(email),
            "phone": c.get("phone"),
            "city": c["address"]["city"],
            "zipcode": c["address"]["zipcode"],
            # converting lat amd long to float
            "latitude": float(c["address"]["geo"]["lat"]),
            "longitude": float(c["address"]["geo"]["lng"]),
            "company": c["company"]["name"],
            "website": c.get("website"),
        })

    return rows

# extracting attributes to rows from get_customers function
rows = extract_attributes(get_customers())

# viewing extracted data in rows using slicing
print(rows[9])

{'id': 10, 'name': 'Clementina DuBuque', 'username': 'Moriah.Stanton', 'email': 'Rey.Padberg@karina.biz', 'email_domain': 'karina.biz', 'phone': '024-648-3804', 'city': 'Lebsackbury', 'zipcode': '31428-2261', 'latitude': -38.2386, 'longitude': 57.2232, 'company': 'Hoeger LLC', 'website': 'ambrose.net'}


In [ ]:
def unique_cities(customers: list[dict]) -> set[str]:
    """Return the set of distinct cities."""
    # loop through each customer in customers
    # only return if city available
    # if city found retrieve it
    return {c["city"] for c in customers if c.get("city")}

cities = unique_cities(rows)

print(cities)
print("Number of cities : ", len(cities))
# cheking no of customers by checking len of rows
# since each row is a unique customer
print("Number of customers : ", len(rows))

{'South Elvis', 'South Christy', 'Aliyaview', 'McKenziehaven', 'Lebsackbury', 'Howemouth', 'Gwenborough', 'Roscoeview', 'Bartholomebury', 'Wisokyburgh'}
Number of cities :  10
Number of customers :  10


In [56]:
def city_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per city using a loop and/or a dict comprehension (no Counter, no pandas)."""
    
    # making dictonary to hold count per city
    counts: dict[str, int] = {}

    #loop through customers
    for c in customers:
        city = c.get("city")
        # if city not found, skip 
        if not city:
            continue
        # return current count or 0 first time a city is found
        # add 1 everytime a city is found
        counts[city] = counts.get(city, 0) + 1
    
    return counts

# city_counts = city_counts(rows)
# print(city_counts)
 

In [44]:
def email_domain_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per email top-level domain (e.g. 'biz', 'tv')."""
    domains = [
        # splitting the domain once at the last "." from the email domain
        c["email_domain"].rsplit(".", 1)[1]
        for c in customers
        # validating email domain exists and has a "." separator
        if c.get("email_domain") and "." in c["email_domain"]
    ]
    return {domain: domains.count(domain) for domain in set(domains)}

# domain_counts = email_domain_counts(rows)
# print(domain_counts)


In [ ]:
# function to sort the domain and city counts
def print_sorted(title: str, counts: dict[str, int]) -> None:
    print(title)
    # looping through the key value pairs 
    # -kv[1] sort by descending order
    # kv[0] sort alphabetically if num is equal
    for key, num in sorted(counts.items(), key = lambda kv: (-kv[1], kv[0])):
        print(f"{key}: {num}")

print_sorted("Customers per city", city_counts(rows))
print()
print_sorted("Customers per email domain", email_domain_counts(rows))


Customers per city
Aliyaview: 1
Bartholomebury: 1
Gwenborough: 1
Howemouth: 1
Lebsackbury: 1
McKenziehaven: 1
Roscoeview: 1
South Christy: 1
South Elvis: 1
Wisokyburgh: 1

Customers per email domain
biz: 3
ca: 1
info: 1
io: 1
me: 1
net: 1
org: 1
tv: 1


In [ ]:
def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    # pass rows to create a data frame
    return pd.DataFrame(rows)

# creating data frame df 
df = to_dataframe(rows)

# viewing top 5 rows in the data frame
df.head()


,id,name,username,email,email_domain,phone,city,zipcode,latitude,longitude,company,website
0,1,Leanne Graham,Bret,Sincere@april.biz,april.biz,1-770-736-8031 x56442,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,hildegard.org
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,melissa.tv,010-692-6593 x09125,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,anastasia.net
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,yesenia.net,1-463-123-4447,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,ramiro.info
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,kory.org,493-170-9623 x156,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,kale.biz
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,annie.ca,(254)954-1289,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,demarco.info


In [ ]:
# viewing data types for each column
df.dtypes

id                int64
name                str
username            str
email               str
email_domain        str
phone               str
city                str
zipcode             str
latitude        float64
longitude       float64
company             str
website             str
dtype: object

In [ ]:
# using value counts to verify count of cities
df["city"].value_counts()

# result matches the output of the city_counts function

city
Gwenborough       1
Wisokyburgh       1
McKenziehaven     1
South Elvis       1
Roscoeview        1
South Christy     1
Howemouth         1
Aliyaview         1
Bartholomebury    1
Lebsackbury       1
Name: count, dtype: int64

In [ ]:
# counting the domain counts in the df
# extracting the domain from email domain
dmn = df["email_domain"].str.rsplit(".", n=1).str[-1]
dmn.value_counts()


email_domain
biz     3
tv      1
net     1
org     1
ca      1
info    1
me      1
io      1
Name: count, dtype: int64

---

## exporting the data to csv files

In [58]:
OUTPUT_DIR = Path("output")

# setting columns for the customers file
CUSTOMER_COLUMNS = [
    "id", "name", "username", "email", "email_domain", "phone", "city",
    "zipcode", "latitude", "longitude", "company", "website",
]

# setting the columns for the city file
CITY_COLUMS = ["city", "count"]

def export_results(df: pd.DataFrame, city_count_dict: dict[str, int]) -> None:
    # creating output dir if it does not exist
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    # setting index False to remove the dataframe index
    df[CUSTOMER_COLUMNS].to_csv(OUTPUT_DIR / "customers.csv", index=False)

    # creating a dataframe from the city counts function
    city_df = (
        pd.DataFrame(list(city_count_dict.items()), columns=CITY_COLUMS)
        .sort_values(["count", "city"], ascending=[False, True])
    )
    # exporting the city dataframe to a csv file
    city_df.to_csv(OUTPUT_DIR / "city_counts.csv", index=False)


export_results(df, city_counts(rows))

In [60]:
def plot_distribution(counts: dict[str, int], path: Path, title: str = "Customer Distribution", ylabel: str = "") -> None:
    """Save a labelled chart of the distribution to path."""
    # sets path where the file will be saved, if path doesnt exist it is created
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # sorting in ascending order 
    items = sorted(counts.items(), key=lambda kv: (kv[1], kv[0]))

    # using list comprehension to extract labels and values from the counts
    # extract label, ignore the value
    labels = [k for k, _ in items]
    # extract value in each tuple, ignore the label
    values = [v for _, v in items]


    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(labels) + 1)))
    # making a horizontal bar chart
    bars = ax.barh(labels, values, color="steelblue")
    ax.bar_label(bars, padding=3)  # print the count at the end of each bar
    ax.set_title(title)
    ax.set_xlabel("Number of customers")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # no 0.5 customers
    ax.set_xlim(0, max(values) * 1.15)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


plot_distribution(
        city_counts(rows),
        Path("output/customer_distribution.png"),
        title="Customers per city",
        ylabel="City",
    )
plot_distribution(
    email_domain_counts(rows),
    Path("output/email_domain_distribution.png"),
    title="Customers per email domain",
    ylabel="Email domain",
)
print("Saved both charts to output/")


Saved both charts to output/


In [ ]:


if __name__ == "__main__":
    # TODO: run each step and print a short findings summary
    pass
